# Model: LightGBM

Owner: **Arman**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from lightgbm import LGBMRegressor

MODEL_NAME = "lightgbm"

In [2]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

(140208, 55) (17520, 55) (17520, 55)


In [3]:
# TEMPERATURE/radiation are same-day actuals so not usable, only the day_before lags are

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]
FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

48

In [4]:
model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
)

model.fit(train[FEATURES], train[TARGET])

print("Training finished")

Training finished


In [5]:
validation_predictions = model.predict(validation[FEATURES])

validation_rmse = np.sqrt(
    mean_squared_error(validation[TARGET], validation_predictions)
)

print(f"Validation RMSE: {validation_rmse:.2f} MW")

Validation RMSE: 425.94 MW


In [6]:
results = []
best_rmse = float("inf")
best_model = None
best_params = None

for trees in [100, 300]:
    for leaves in [15, 31]:
        candidate = LGBMRegressor(
            n_estimators=trees,
            num_leaves=leaves,
            learning_rate=0.1,
            random_state=42,
            verbosity=-1,
        )

        candidate.fit(train[FEATURES], train[TARGET])
        val_predictions = candidate.predict(validation[FEATURES])

        rmse = np.sqrt(
            mean_squared_error(validation[TARGET], val_predictions)
        )

        results.append({
            "trees": trees,
            "leaves": leaves,
            "validation_rmse": rmse,
        })

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = candidate
            best_params = {"trees": trees, "leaves": leaves}

model = best_model

display(pd.DataFrame(results).sort_values("validation_rmse"))
print("Selected settings:", best_params)

,trees,leaves,validation_rmse
3,300,31,418.745182
2,300,15,420.280441
1,100,31,425.939019
0,100,15,434.724854


Selected settings: {'trees': 300, 'leaves': 31}


In [7]:
predictions = model.predict(test[FEATURES])
print("Number of predictions:", len(predictions))

Number of predictions: 17520


In [9]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'lightgbm', 'rmse': np.float64(463.9951386111268), 'mae': 307.403003724203, 'mape_pct': 3.74458716611304, 'r2': 0.8621784126576358}


In [10]:
# saving prediction performance

pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2
0,random_forest,493.696778,322.451412,3.927903,0.843969
1,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354
2,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980
3,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319
4,lightgbm,463.995139,307.403004,3.744587,0.862178


**LightGBM model summary**

LightGBM was trained on the shared 2010-2017 data using 48 features, including previous-day demand, temperature and radiation, plus calendar features. Four combinations of tree count and leaf count were compared using 2018 validation data. The best combination was 300 trees with up to 31 leaves per tree and a learning rate of 0.1, giving a validation RMSE of 418.75 MW.

With these settings fixed, the model achieved an RMSE of 464.00 MW, MAE of 307.40 MW, MAPE of 3.74% and R² of 0.8622 on the 2019 test data. This means its predictions were about 307 MW away from actual demand on average. These results support including LightGBM in the team's model comparison, but do not establish performance during the excluded COVID period.